# Шаг 4. Пульт управления: ползунки меняют картинку в реальном времени

Параметры фильтров (порог, размер окна, контраст) редко угадываются с первого раза.
Удобнее всего подбирать их **интерактивно**: двигать ползунок и сразу видеть результат.
В OpenCV для этого есть **трекбары** (`cv2.createTrackbar`) — ползунки в окне программы.

> ⚠️ Этот ноутбук открывает **отдельное окно OpenCV** (не внутри Jupyter). Оно может появиться позади браузера.
> Пока окно открыто, ячейка выполняется (значок `[*]`). Выход — клавиша **q** или **Esc** (окно должно быть активно) либо закрыть окно крестиком.
> В Google Colab и на удалённом сервере такие окна не работают: там откройте `04_trackbars_colab.ipynb`.

**Режимы** (ползунок `mode`): 0 — цвет, 1 — серый, 2 — негатив, 3 — порог, 4 — контуры Canny.

**Структура ноутбука (алгоритм):**
1. Импортируем OpenCV и помощники, загружаем фото (уменьшаем до 960 × 540).
2. Создаём окно и пять ползунков: яркость, контраст, размытие, режим, порог.
3. Запускаем **бесконечный цикл обработки**:
   1. читаем текущие положения всех ползунков;
   2. применяем к **исходной** картинке яркость и контраст, затем размытие;
   3. в зависимости от режима превращаем результат в серый / негатив / порог / контуры;
   4. показываем кадр и ждём 30 мс нажатия клавиши;
   5. если нажата q/Esc или окно закрыто — выходим из цикла.
4. Закрываем все окна.

## 1. Импорт и загрузка картинки

`fit(img, 960, 540)` уменьшает фото так, чтобы оно влезло в 960 × 540, сохраняя пропорции.
Небольшая картинка обрабатывается быстрее — ползунки реагируют без задержек.

In [ ]:
import cv2

from common import PIPE, fit, load

img = fit(load(PIPE), 960, 540)

## 2. Окно и ползунки

`cv2.namedWindow(WINDOW)` создаёт пустое окно с именем `"controls"` — к нему будут привязаны ползунки.

`cv2.createTrackbar(имя, окно, начальное значение, максимум, функция)` добавляет ползунок от 0 до максимума.
Ползунки OpenCV умеют только **целые неотрицательные** числа, поэтому:
- **brightness**: 0…200, а в коде вычитаем 100 → получаем сдвиг от −100 до +100 (100 = без изменений);
- **contrast x10**: 0…30, а в коде делим на 10 → множитель от 0.0 до 3.0 (10 = ×1.0);
- **blur**: 0…15 — номер размера окна (0 = без размытия);
- **mode**: 0…4 — режим показа;
- **threshold**: 0…255 — порог для режимов 3 и 4.

Последний аргумент — функция, которую OpenCV вызывает при каждом сдвиге ползунка.
Мы читаем значения сами в цикле, поэтому передаём «пустышку» `nothing`.

In [ ]:
WINDOW = "controls"
cv2.namedWindow(WINDOW)


def nothing(_):
    pass


cv2.createTrackbar("brightness", WINDOW, 100, 200, nothing)   # 100 = без изменений
cv2.createTrackbar("contrast x10", WINDOW, 10, 30, nothing)   # 10 = x1.0
cv2.createTrackbar("blur", WINDOW, 0, 15, nothing)
cv2.createTrackbar("mode", WINDOW, 0, 4, nothing)
cv2.createTrackbar("threshold", WINDOW, 120, 255, nothing)

## 3. Главный цикл

Каждую итерацию цикл заново строит картинку **с нуля из исходной** `img` — поэтому эффекты не накапливаются,
а сдвиг ползунка назад возвращает прежний вид.

Порядок обработки:
1. `cv2.getTrackbarPos` — читаем положения ползунков и переводим их в настоящие параметры.
2. `convertScaleAbs(img, alpha, beta)` — контраст и яркость: `v · alpha + beta`, с обрезкой до 0…255.
3. Размытие Гаусса: размер окна должен быть нечётным, поэтому из номера `k` делаем `2k + 1` (1 → 3, 2 → 5, …).
4. Режим: серый (`cvtColor`), негатив (`255 - out`), порог (`threshold` с порогом `t`) или контуры
   (`Canny` с порогами `t/2` и `t` — у Canny нижний порог обычно вдвое меньше верхнего).
5. `cv2.imshow` показывает результат, `cv2.waitKey(30)` ждёт клавишу 30 мс (≈ 33 кадра в секунду)
   и заодно даёт окну перерисоваться. `& 0xFF` оставляет только код клавиши (младший байт).
6. Выход: клавиша `q`, `Esc` (код 27) или окно закрыто крестиком (`WND_PROP_VISIBLE < 1`).

In [ ]:
while True:
    beta = cv2.getTrackbarPos("brightness", WINDOW) - 100
    alpha = cv2.getTrackbarPos("contrast x10", WINDOW) / 10
    k = cv2.getTrackbarPos("blur", WINDOW)
    mode = cv2.getTrackbarPos("mode", WINDOW)
    t = cv2.getTrackbarPos("threshold", WINDOW)

    out = cv2.convertScaleAbs(img, alpha=alpha, beta=beta)
    if k > 0:
        out = cv2.GaussianBlur(out, (2 * k + 1, 2 * k + 1), 0)  # размер окна должен быть нечётным

    gray = cv2.cvtColor(out, cv2.COLOR_BGR2GRAY)
    if mode == 1:
        out = gray
    elif mode == 2:
        out = 255 - out
    elif mode == 3:
        _, out = cv2.threshold(gray, t, 255, cv2.THRESH_BINARY)
    elif mode == 4:
        out = cv2.Canny(gray, t // 2, t)

    cv2.imshow(WINDOW, out)
    key = cv2.waitKey(30) & 0xFF
    if key in (ord("q"), 27) or cv2.getWindowProperty(WINDOW, cv2.WND_PROP_VISIBLE) < 1:
        break

cv2.destroyAllWindows()

## Попробуйте сами

1. Подберите в режиме 3 (порог) такие яркость, контраст и порог, при которых маркировка читается лучше всего. Запишите значения.
2. Сравните режим 4 (контуры) с размытием 0 и с размытием 3. Почему с размытием контуров меньше?
3. Добавьте шестой режим `5` — поворот на 180° (`cv2.rotate`). Не забудьте поменять максимум ползунка `mode`.
4. Добавьте ползунок для выбора типа размытия: Гаусс или медиана.